# Efeito dos hiperparâmetros do Kim (na T4 / GPU)

Roda o nosso multiclasse `content_ext` com os **parâmetros exatos do Kim** (1000 árvores,
lr 0,05, depth 6, subsample/colsample 0,8, L2=1,0) — em **GPU** para ser rápido — nos **dois
splits** (aleatório e temporal por arquivo), e compara com os nossos parâmetros atuais
(300 árvores, lr 0,3, depth 8): random **0,9936** / temporal **0,9658**.

**Ative a GPU:** Ambiente de execução → Alterar tipo → **T4 GPU**.

In [ ]:
# Setup: clona o repo + baixa features (LFS, blindado)
import os
REPO = 'someip-ids-multiclass-contentext'
if not os.path.exists('data/ours_ext/X.npz'):
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')
    elif os.path.isdir(REPO):
        os.chdir(REPO)
    else:
        os.system('apt-get -qq install -y git-lfs >/dev/null 2>&1; git lfs install')
        os.system(f'git clone -q https://github.com/GuilhermeFrick/{REPO}.git')
        os.chdir(REPO); os.system('git lfs pull')
if os.path.exists('data/ours_ext/X.npz') and os.path.getsize('data/ours_ext/X.npz') < 100000:
    os.system('git lfs install; git lfs pull')
import xgboost; print('xgboost', xgboost.__version__, '| cwd:', os.getcwd())

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from xgboost import XGBClassifier

CLASSES = ['normal','dos','fuzzy','mitm_single','mitm_multi']; N=len(CLASSES); SEED=0
CONTENT_EXT = list(range(12)) + [12, 13, 14, 16]
FILE_COUNTS = [('benign',2193802),('dos',1864530),('fuzzy1',2197113),('fuzzy2',1304154),
               ('fuzzy3',2223650),('mitm_multi',2412529),('mitm_single',2037576)]
OURS_REF = {'random': 0.9936, 'temporal': 0.9658}

X = np.load('data/ours_ext/X.npz')['a'][:, CONTENT_EXT]
y = np.load('data/ours_ext/y_multi.npz')['a']
print('X:', X.shape)

# Params do Kim (Tabela 2) — em GPU (device='cuda'). Se não houver GPU, troque para device='cpu'.
def kim_xgb():
    return XGBClassifier(objective='multi:softprob', num_class=N, n_estimators=1000, max_depth=6,
                         learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
                         tree_method='hist', device='cuda', n_jobs=-1, eval_metric='mlogloss')

def rep(tag, y_te, y_pred):
    r = classification_report(y_te, y_pred, target_names=CLASSES, digits=4, output_dict=True)
    print(f'\n== {tag} ==')
    for c in CLASSES: print(f'  {c:12} F1={r[c]["f1-score"]:.4f}')
    print(f'  macro-F1={r["macro avg"]["f1-score"]:.4f}  accuracy={r["accuracy"]:.4f}')
    return r['macro avg']['f1-score']

In [ ]:
# Split ALEATÓRIO (params Kim)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, random_state=SEED, stratify=y)
m = kim_xgb(); m.fit(Xtr, ytr)
mf_rand = rep('ALEATÓRIO (params Kim)', yte, m.predict(Xte))

In [ ]:
# Split TEMPORAL por arquivo (params Kim)
tr, te, s = [], [], 0
for _, c in FILE_COUNTS:
    cut = s + int(c*0.7); tr.append(np.arange(s, cut)); te.append(np.arange(cut, s+c)); s += c
itr, ite = np.concatenate(tr), np.concatenate(te)
m2 = kim_xgb(); m2.fit(X[itr], y[itr])
mf_temp = rep('TEMPORAL por arquivo (params Kim)', y[ite], m2.predict(X[ite]))

In [ ]:
# Comparação params Kim vs nossos params
import pandas as pd
df = pd.DataFrame({
    'params Kim (1000/0.05/d6/sub0.8)': {'random': mf_rand, 'temporal': mf_temp},
    'nossos params (300/0.3/d8)':       OURS_REF,
}).round(4)
display(df)

## Leitura

Compare os macro-F1: se ficarem **próximos dos nossos** (random ≈ 0,99 / temporal ≈ 0,96), isso
**reforça a defesa** — o resultado é **robusto à escolha de hiperparâmetros** (não é artefato de
um config 'agressivo'), e os params mais regularizados do Kim não mudam a conclusão. O número
honesto continua sendo o **temporal**; a generalização a ataque novo, o **zero-day (~0,60)**.